In [4]:
import sys
from pathlib import Path
from uuid import uuid4
import pandas as pd
from playwright.async_api import async_playwright

PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# 使用visual安装时已有的显示服务配置。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
context = await browser.new_context(
    viewport={"width": 1000, "height": 620}, accept_downloads=True
)
page = await context.new_page()
page.set_default_timeout(20000)  # 配置默认超时，单位毫秒，不需要await。


In [12]:
# ① 未登录时打开QQ邮箱；已扫码的当前页面直接用于验证。
if not page.url.startswith("https://wx.mail.qq.com/home/"):
    await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
    print("请在浏览器画面中扫码", flush=True)

# ② 当前网页扫码后直接进入收件箱，当前激活的侧栏项已在真实页面核实。
# 等待当前收件箱侧栏项；后续批次循环可安全点击它返回收件箱。
inbox = page.locator(".frame-sidebar-menu.sidebar-menu-active")

# ③ 最长等待10分钟。
await inbox.wait_for(state="visible", timeout=600000)
print("登录成功", flush=True)


登录成功


In [9]:
# 以下选择器已用实际 QQ 邮箱页面和 Chromium DevTools 接口核实。
ROW_CSS = ".mail-list-page-item[data-mailid]"
SENDER_CSS = ".cmp-account-nick"
AD_CSS = ".gg-unread-count"
SUBJECT_CSS = ".mail-subject"
TIME_CSS = ".mail-time"
HOVER_CSS = ".cmp-account-nick"
EMAIL_CSS = ".xmail-cmp-account-card .cmp-email-address"


In [7]:
async def get_ids():
    await page.locator(ROW_CSS).first.wait_for()
    # 一次读取整批ID，避免列表更新时逐个保存旧元素。
    ids = await page.locator(ROW_CSS).evaluate_all(
        "rows => rows.map(row => row.getAttribute('data-mailid'))"
    )
    if not ids or not all(ids):
        raise ValueError("没有取得有效邮件ID，请检查页面")
    return ids


def get_row(mail_id):
    return page.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]')


async def read_list_mail(mail_id):
    row = get_row(mail_id)
    if await row.locator(AD_CSS).count():
        return None
    sender = await row.locator(SENDER_CSS).inner_text()
    subject = await row.locator(SUBJECT_CSS).inner_text()
    time_text = await row.locator(TIME_CSS).inner_text()

    # 先移开鼠标，等待上一张卡片消失。
    await row.locator(TIME_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="hidden")
    # 先把发件人滚入可见区域。滚动时页面会暂时停止响应悬停。
    await row.locator(HOVER_CSS).scroll_into_view_if_needed()
    await page.locator(".mail-list-page .ui-float-scroll-body-scrolling").wait_for(state="hidden")
    # 滚动结束后，再悬停本封发件人，等待卡片显示。
    await row.locator(HOVER_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="visible")
    email = await page.locator(EMAIL_CSS).inner_text()
    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}


In [10]:
basic_records = []
for mail_id in await get_ids():
    record = await read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))


成功采集： 10


In [11]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)


CSV记录数： 10


In [13]:
# 真实列表滚动容器已用 DevTools 的 scrollHeight/clientHeight 核实。
SCROLL_CSS = ".mail-list-page .ui-float-scroll-body"


def get_box():
    return page.locator(SCROLL_CSS)


async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # 第四阶段定义正文函数；本阶段False，不执行这里。
            record.update(await read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


async def collect_batches(include_details=False, max_mails=0):
    await inbox.click()
    await page.locator(ROW_CSS).first.wait_for()
    await get_box().evaluate("box => box.scrollTop = 0")
    records, seen_ids = [], set()
    for batch in range(3):
        top = await get_box().evaluate("box => box.scrollTop")
        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break
        box = get_box()
        height = await box.evaluate("box => box.clientHeight")
        full_height = await box.evaluate("box => box.scrollHeight")
        next_top = min(top + height, full_height - height)
        if next_top <= top:
            print("已到列表末尾")
            break
        old_ids = await get_ids()
        await box.evaluate("(box, value) => box.scrollTop = value", next_top)
        # 在浏览器中比较新旧ID，改变后才继续。
        await page.wait_for_function("""data => {
            const ids = Array.from(document.querySelectorAll(data.selector),
                                   node => node.getAttribute('data-mailid'));
            return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);
        }""", arg={"selector": ROW_CSS, "ids": old_ids})
    return records, seen_ids


In [14]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())


第1批，新增25，累计25


第2批，新增18，累计43


第3批，新增19，累计62
CSV记录数： 62
唯一ID数： 62
重复ID数： 0


In [15]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = await parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())


重复采集新增： 0
CSV内容未变： True


In [16]:
# 正文容器与无文字返回图标均已在真实邮件详情页核实。
BODY_CSS = ".mail-detail-content .qmbox"
BACK_CSS = ".mail-list-page-toolbar .left-wrap > .xmail-ui-btn"
DOWNLOAD_ATTACHMENTS = False


async def read_detail(mail_id):
    top = await get_box().evaluate("box => box.scrollTop")
    row = get_row(mail_id)
    await row.locator(SUBJECT_CSS).click()
    try:
        body_node = page.locator(BODY_CSS)
        await body_node.wait_for(state="visible")
        body = (await body_node.inner_text()).strip()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            files = await download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        await page.locator(BACK_CSS).click()
        await get_box().wait_for()
        await get_box().evaluate("(box, value) => box.scrollTop = value", top)
        await get_row(mail_id).wait_for(state="attached")


In [17]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]
trial_records, _ = await collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))


In [18]:
detail_records, _ = await collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())


第1批，新增25，累计25


第2批，新增18，累计43


第3批，新增19，累计62
正文CSV记录数： 62
重复ID数： 0
